# E05

In [ ]:

from __future__ import annotations

import csv
import json
import sys
from copy import deepcopy
from pathlib import Path


import shutil
import tempfile
import zipfile

SOURCE_ZIP_NAME = 'syniscopy_source.zip'
IN_COLAB = Path('/content').exists()
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
if IN_COLAB and not DRIVE_MYDRIVE.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive was not mounted automatically:', repr(exc))


def find_supplemental_root() -> Path:
    explicit = globals().get('SYNISCOPY_SUPPLEMENTAL_ROOT', None)
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    if DRIVE_MYDRIVE.exists():
        candidates.extend([
            DRIVE_MYDRIVE / 'supplemental',
            DRIVE_MYDRIVE / 'SyniscopySupplemental',
        ])
    candidates.extend([
        Path('/content/supplemental'),
        Path('/content/SyniscopySupplemental'),
    ])
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if (resolved / 'supplemental' / 'E01.ipynb').exists():
            resolved = resolved / 'supplemental'
        if (resolved / 'E01.ipynb').exists() and (
            (resolved / SOURCE_ZIP_NAME).exists() or (resolved.parent / 'codebase').is_dir()
        ):
            return resolved
    raise RuntimeError(
        'Syniscopy supplemental folder not found. Upload the supplemental folder to Drive as MyDrive/supplemental, '
        'including syniscopy_source.zip.'
    )


def prepare_syniscopy_source(supplemental_root: Path) -> Path:
    source_root = supplemental_root.parent
    if (source_root / 'codebase').is_dir():
        return source_root
    zip_path = supplemental_root / SOURCE_ZIP_NAME
    if not zip_path.exists():
        raise FileNotFoundError(
            f'Missing {zip_path}. Run python supplemental/package_experiments_for_colab.py before uploading supplemental/.'
        )
    extract_dir = Path('/content/syniscopy_source') if IN_COLAB else Path(tempfile.gettempdir()) / 'syniscopy_source'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    if (extract_dir / 'codebase').is_dir():
        return extract_dir
    candidates = [p for p in extract_dir.iterdir() if p.is_dir() and (p / 'codebase').is_dir()]
    if len(candidates) != 1:
        raise RuntimeError(f'Could not find a single codebase/ root after extracting {zip_path}.')
    return candidates[0]


SUPPLEMENTAL_ROOT = find_supplemental_root()
ROOT = prepare_syniscopy_source(SUPPLEMENTAL_ROOT)
CODEBASE = ROOT / 'codebase'
OUTPUT_DIR = SUPPLEMENTAL_ROOT / 'outputs' / 'E05'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(CODEBASE) not in sys.path:
    sys.path.insert(0, str(CODEBASE))

import numpy as np

from config import PARAMS
from supervision_policy import SupervisionAudit, SupervisionPolicy

PIXEL_SIZE_NM = 65.0


size = 64
yy, xx = np.indices((size, size))
rr2 = (xx - size // 2) ** 2 + (yy - size // 2) ** 2
geometry = (rr2 <= 7 ** 2).astype(np.uint8) * 255
strong = 40.0 * np.exp(-rr2 / (2.0 * 4.0 ** 2))
weak = 0.06 * np.exp(-rr2 / (2.0 * 4.0 ** 2))
zero = np.zeros_like(strong)

params = deepcopy(PARAMS)
params.update({
    'pixel_size_nm': PIXEL_SIZE_NM,
    'image_size_pixels': size,
    'supervision_target': 'mask_supported',
    'supervision_support_factors': ['signal', 'information'],
    'supervision_supported_threshold': 0.2,
    'supervision_temporal_support_enabled': False,
    'supervision_signal_support_enabled': True,
    'supervision_information_support_enabled': True,
    'supervision_crlb_xy_max_nm': PIXEL_SIZE_NM,
    'background_intensity': 1.0,
    'read_noise_counts': 1.0,
    'camera_gain_e_per_count': 1.0,
    'shot_noise_enabled': True,
    'gaussian_noise_enabled': True,
})
policy = SupervisionPolicy(params, num_particles=1)
audit = SupervisionAudit()
for frame_index, contrast in enumerate((strong, weak, zero)):
    result = policy.evaluate(
        particle_index=0,
        frame_index=frame_index,
        position_nm=np.array([0.0, 0.0, 0.0], dtype=float),
        contrast_image=contrast,
        geometry_mask=geometry,
    )
    audit.add(result['record'])
summary = audit.summary()
summary_json = OUTPUT_DIR / 'supervision_audit_summary.json'
summary_json.write_text(json.dumps(summary, indent=2, sort_keys=True, allow_nan=False), encoding='utf-8')

rows = []
for key, value in sorted(summary.items()):
    if isinstance(value, dict):
        for nested_key, nested_value in sorted(value.items()):
            rows.append({'metric': f'{key}.{nested_key}', 'value': nested_value})
    else:
        rows.append({'metric': key, 'value': value})
summary_csv = OUTPUT_DIR / 'supervision_audit_summary.csv'
with summary_csv.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['metric', 'value'])
    writer.writeheader()
    writer.writerows(rows)
print('wrote raw E05 supervision audit data:', OUTPUT_DIR)
